# 🧪 Coral Reef Disease Detection — Training on Google Colab

**What this notebook does:**
1. Clones the **official YOLOv5** repository (provides the full AI engine)
2. Downloads the **coral reef dataset** from the MAFFN research repo
3. Configures the dataset for training (5 disease classes)
4. Trains the YOLOv5 model on the coral reef images using the T4 GPU
5. Downloads the trained model (`best.pt`) back to your computer

---
### Step 1: Check GPU Connection
Verify that Google Colab has assigned an NVIDIA GPU.

In [1]:
!nvidia-smi

Wed Feb 25 13:48:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2: Clone Official YOLOv5 + Download Coral Dataset
We clone the **official YOLOv5** repo (which has all the Python source code the training scripts need), then clone the MAFFN research repo to get the coral reef dataset.

In [2]:
# Clone official YOLOv5 (the full AI engine)
!git clone https://github.com/ultralytics/yolov5.git

# Clone the MAFFN repo (for the coral reef dataset only)
!git clone https://github.com/sivamani-k/MAFFN_YOLOv5-based-Coral-reefs-health-detection-and-coral-reefs-dataset.git maffn_repo

# Move into the YOLOv5 directory
import os
os.chdir('/content/yolov5')

# Install YOLOv5 dependencies
%pip install -r requirements.txt

print('\n=== Setup Complete ===')

Cloning into 'yolov5'...
remote: Enumerating objects: 17822, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 17822 (delta 14), reused 6 (delta 6), pack-reused 17798 (from 4)
Receiving objects: 100% (17822/17822), 16.99 MiB | 20.56 MiB/s, done.
Resolving deltas: 100% (12147/12147), done.
Cloning into 'maffn_repo'...
remote: Enumerating objects: 4654, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 4654 (delta 9), reused 29 (delta 0), pack-reused 4611 (from 1)
Receiving objects: 100% (4654/4654), 137.09 MiB | 34.66 MiB/s, done.
Resolving deltas: 100% (13/13), done.
Updating files: 100% (4747/4747), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 14.8 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalli

### Step 3: Prepare the Coral Reef Dataset
Copy the dataset into the YOLOv5 directory and create a proper `data.yaml` config file with absolute paths so YOLOv5 can find the images.

In [8]:
import shutil
import yaml

# ==============================
# 1️⃣ Copy Dataset
# ==============================
src = '/content/maffn_repo/Dataset with annotation in YOLO format'
dst = '/content/yolov5/coral_dataset'

if not os.path.exists(dst):
    shutil.copytree(src, dst)
    print(f'Dataset copied to: {dst}')
else:
    print(f'Dataset already exists at: {dst}')


# ==============================
# 2️⃣ Fix Folder Names (annotations → labels)
# ==============================
for split in ['train', 'valid', 'test']:
    old = f'{dst}/{split}/annotations'
    new = f'{dst}/{split}/labels'

    if os.path.exists(old) and not os.path.exists(new):
        os.rename(old, new)
        print(f'Renamed: {old} → {new}')


# ==============================
# 3️⃣ Create data.yaml (ABSOLUTE paths)
# ==============================
data_config = {
    'train': f'{dst}/train/images',
    'val': f'{dst}/valid/images',
    'test': f'{dst}/test/images',
    'nc': 5,
    'names': [
        'Band disease',
        'Bleached disease',
        'Dead Coral',
        'Healthy Coral',
        'White Pox Disease'
    ]
}

yaml_path = '/content/yolov5/coral_data.yaml'

with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print('\n=== Dataset Config (coral_data.yaml) ===')
with open(yaml_path, 'r') as f:
    print(f.read())


# ==============================
# 4️⃣ Verify Dataset Structure
# ==============================
print('=== Dataset Structure ===')

for split in ['train', 'valid', 'test']:
    img_path = f'{dst}/{split}/images'
    lbl_path = f'{dst}/{split}/labels'

    n_imgs = len(os.listdir(img_path)) if os.path.exists(img_path) else 0
    n_lbls = len(os.listdir(lbl_path)) if os.path.exists(lbl_path) else 0

    print(f'  {split}: {n_imgs} images, {n_lbls} labels')

Dataset already exists at: /content/yolov5/coral_dataset
Renamed: /content/yolov5/coral_dataset/train/annotations → /content/yolov5/coral_dataset/train/labels
Renamed: /content/yolov5/coral_dataset/valid/annotations → /content/yolov5/coral_dataset/valid/labels
Renamed: /content/yolov5/coral_dataset/test/annotations → /content/yolov5/coral_dataset/test/labels

=== Dataset Config (coral_data.yaml) ===
names:
- Band disease
- Bleached disease
- Dead Coral
- Healthy Coral
- White Pox Disease
nc: 5
test: /content/yolov5/coral_dataset/test/images
train: /content/yolov5/coral_dataset/train/images
val: /content/yolov5/coral_dataset/valid/images

=== Dataset Structure ===
  train: 2067 images, 2067 labels
  valid: 197 images, 197 labels
  test: 99 images, 99 labels


### Step 4: Start Training! 🚀
This trains the YOLOv5s model on the coral reef dataset.

- `--img 640` : Image size (standard)
- `--batch 16` : Images processed at once (safe for T4 GPU)
- `--epochs 50` : Number of full passes through the dataset
- `--data coral_data.yaml` : Our coral reef dataset config
- `--weights yolov5s.pt` : Start from a pre-trained YOLOv5 small model (transfer learning)
- `--name coral_run` : Name for this training run

> **Note:** 50 epochs should take ~30-60 minutes on a T4 GPU. For maximum accuracy, increase to 100-300 epochs.

In [9]:
!python train.py --img 640 --batch 16 --epochs 50 --data coral_data.yaml --weights yolov5s.pt --name coral_run --cache

wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2026-02-25 14:14:01.910460: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772028841.931174    6607 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772028841.938004    6607 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772028841.956912    6607 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772028841.956937    6607 computation_placer.cc:177] computation placer already registere

### Step 5: View Training Results 📊
Display the training metrics and sample predictions.

In [ ]:
from IPython.display import Image, display
import glob

# Show training results
results_img = glob.glob('runs/train/coral_run*/results.png')
if results_img:
    display(Image(filename=results_img[-1], width=800))
    print(f'Results from: {results_img[-1]}')

# Show sample predictions on validation set
val_imgs = glob.glob('runs/train/coral_run*/val_batch0_pred.jpg')
if val_imgs:
    display(Image(filename=val_imgs[-1], width=800))
    print(f'Predictions from: {val_imgs[-1]}')

### Step 6: Download the Trained Model 📥
Download `best.pt` (the trained AI brain) to your laptop. Once you have this file, you can run coral disease detection locally without needing the GPU!

In [11]:
from google.colab import files
import glob

# Find the best weights file
best_weights = glob.glob('runs/train/coral_run*/weights/best.pt')

if best_weights:
    latest = best_weights[-1]
    print(f'Found trained model at: {latest}')
    
    # Download!
    files.download(latest)
    print('\nDownload started! Check your browser downloads.')
    print('Once downloaded, place best.pt in your project folder.')
else:
    print('Could not find the trained weights file.')
    print('Did the training in Step 4 complete successfully?')
    print('\nAvailable runs:')
    !ls -la runs/train/ 2>/dev/null || echo 'No training runs found'

Found trained model at: runs/train/coral_run3/weights/best.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download started! Check your browser downloads.
Once downloaded, place best.pt in your project folder.


Final Step Download  Trained model 

In [12]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy trained model to Google Drive
import shutil, glob

best = glob.glob('runs/train/coral_run*/weights/best.pt')
last = glob.glob('runs/train/coral_run*/weights/last.pt')

if best:
    shutil.copy2(best[-1], '/content/drive/MyDrive/best.pt')
    print('✅ best.pt copied to Google Drive!')
if last:
    shutil.copy2(last[-1], '/content/drive/MyDrive/last.pt')
    print('✅ last.pt copied to Google Drive!')

print('\n📂 Go to https://drive.google.com → My Drive → download best.pt')


Mounted at /content/drive
✅ best.pt copied to Google Drive!
✅ last.pt copied to Google Drive!

📂 Go to https://drive.google.com → My Drive → download best.pt


Disconnect with collab

In [ ]:
# ✅ AUTO-DISCONNECT: Stops the GPU to save compute credits
import time
print("⏳ Saving results is done. Disconnecting runtime in 30 seconds...")
print("   (Cancel this cell to stay connected)")
time.sleep(30)

from google.colab import runtime
runtime.unassign()  # Disconnects the GPU runtime
print("✅ Runtime disconnected! No more credits being used.")


: 